<a href="https://colab.research.google.com/github/Mustafizb24/gold-directional-forecasting/blob/main/03_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

#Loading the cleaned features dataset from notebook 1

df = pd.read_csv('/content/drive/MyDrive/gold_features.csv', index_col=0, parse_dates = True)

#Checking size, start date and the percentage of up days
print(df.shape)
print(df.index.min())
print(df["target"].mean())

Mounted at /content/drive
(3974, 12)
2010-03-16 00:00:00
0.5266733769501761


In [2]:
from sklearn.model_selection import TimeSeriesSplit #Main tool for walk-forward splits

# The 8 features used as model inputs (raw prices are left out)
feature_cols = ["gold_ret", "dxy_ret", "yield_chg", "gold_ret_lag1", "gold_ret_lag2", "gold_ma20_gap", "gold_ma50_gap", "gold_vol20"]

# x = inputs, y = what we're predicting
x = df[feature_cols]
y = df["target"]

#Walk-forward: 5 splits, each one trains on all past data and tests on the next block of time
tscv = TimeSeriesSplit(n_splits = 5)

# Checking the date range of each split
for i, (train_idx, test_idx) in enumerate(tscv.split(x)):
  train_start = x.index[train_idx[0]].date()
  train_end = x.index[train_idx[-1]].date()
  test_start = x.index[test_idx[0]].date()
  test_end = x.index[test_idx[-1]].date()

  print("Split", i+1, " |  train:", train_start, "to", train_end,
        " |  test:", test_start, "to", test_end )

Split 1  |  train: 2010-03-16 to 2012-10-31  |  test: 2012-11-01 to 2015-06-19
Split 2  |  train: 2010-03-16 to 2015-06-19  |  test: 2015-06-22 to 2018-02-08
Split 3  |  train: 2010-03-16 to 2018-02-08  |  test: 2018-02-09 to 2020-09-25
Split 4  |  train: 2010-03-16 to 2020-09-25  |  test: 2020-09-28 to 2023-05-15
Split 5  |  train: 2010-03-16 to 2023-05-15  |  test: 2023-05-16 to 2025-12-30


In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

#Pipeline: scaling features first, then run logistic regression
lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1000))
])

print(lr_model)

Pipeline(steps=[('scaler', StandardScaler()),
                ('lr', LogisticRegression(max_iter=1000))])


In [4]:
# Installing CodeCarbon (energy and carbon tracking)
!pip install codecarbon -q

from codecarbon import EmissionsTracker

# Tracker for this notebook's model training
tracker = EmissionsTracker(project_name="logistic_regression", log_level="error")

[codecarbon WARNING @ 17:31:02] Multiple instances of codecarbon are allowed to run at the same time.


In [8]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Lists to collect results
split_results = [] # One row of scores per split
all_preds = [] # Every test day's prediction

tracker.start()

for i, (train_idx, test_idx) in enumerate(tscv.split(x)):
  # Training and test data for this split
  x_train, x_test = x.iloc[train_idx], x.iloc[test_idx]
  y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

  # Train on this split's past data only
  lr_model.fit(x_train, y_train)

  # Predict the test period: up/down, and probability of up
  y_pred = lr_model.predict(x_test)
  y_prob = lr_model.predict_proba(x_test)[:, 1]

  # Score this split
  split_results.append({
      "split": i+1,
      "accuracy": accuracy_score(y_test, y_pred),
      "f1": f1_score(y_test, y_pred),
      "roc_auc": roc_auc_score(y_test, y_prob)
  })

  # Keep each test day's prediction
  all_preds.append(pd.DataFrame({"actual": y_test, "lr_pred": y_pred, "lr_prob": y_prob}))

emissions = tracker.stop()
print("Training done. Emissions (kg C02):", emissions)

Training done. Emissions (kg C02): 1.1626891050704476e-06


In [9]:
# Results table: one row per split
lr_results = pd.DataFrame(split_results)
print(lr_results.round(3))

# Average across the 5 splits
print("\nAverage:")
print(lr_results[["accuracy", "f1", "roc_auc"]].mean().round(3))

# Compare with the majority-class baseline
print("\nBeats 51.8% baseline?", lr_results["accuracy"].mean() > 0.518)

   split  accuracy     f1  roc_auc
0      1     0.488  0.624    0.532
1      2     0.512  0.622    0.515
2      3     0.527  0.623    0.531
3      4     0.533  0.597    0.514
4      5     0.547  0.628    0.536

Average:
accuracy    0.521
f1          0.619
roc_auc     0.525
dtype: float64

Beats 51.8% baseline? True
